# dashboard model difference explanations v1

目标：验证 `src/dashboard` 可以从统一 dashboard export 读取 `model_difference_explanations`，并在 view model 与静态 HTML 中展示模型 winner/loser、regime/window 差异和 review action。

边界：本实验复用 QuantBench smoke dashboard 与上一轮 analysis 解释表，只验证 dashboard 接入，不判断模型是否可晋升。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import build_dashboard_view_model, load_model_matrix_dashboard, render_dashboard_html, write_dashboard_html

CARD = REPO_ROOT / 'research/dashboard_model_difference_explanations_v1'
SOURCE_CARD = REPO_ROOT / 'research/quantbench_model_matrix_smoke_v1'
DIFF_CARD = REPO_ROOT / 'research/analysis_model_difference_explanations_v1'
CARD.mkdir(parents=True, exist_ok=True)

payload = load_model_matrix_dashboard(SOURCE_CARD, top_n=12)
difference_rows = pd.read_csv(DIFF_CARD / 'model_difference_explanations.csv').to_dict(orient='records')
payload['model_difference_explanations'] = difference_rows
payload['artifacts']['model_difference_explanations'] = str((DIFF_CARD / 'model_difference_explanations.csv').relative_to(REPO_ROOT))
(CARD / 'dashboard_export.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
len(payload['top_models']), len(difference_rows), payload['artifacts']['model_difference_explanations']

(12,
 19,
 'research/analysis_model_difference_explanations_v1/model_difference_explanations.csv')

In [2]:
view = build_dashboard_view_model(payload)
(CARD / 'dashboard_view_model.json').write_text(json.dumps(view, ensure_ascii=False, indent=2), encoding='utf-8')
html = write_dashboard_html(view, output_path=CARD / 'dashboard.html', title='SOXL Model Difference Dashboard')

summary = {
    'schema_version': 'research.dashboard_model_difference_explanations.v1',
    'status': 'completed',
    'source_card': str(SOURCE_CARD.relative_to(REPO_ROOT)),
    'difference_source_card': str(DIFF_CARD.relative_to(REPO_ROOT)),
    'model_card_count': int(len(view['model_cards'])),
    'difference_explanation_count': int(view['model_difference_summary']['count']),
    'top_explanation_signal': view['model_difference_summary']['top_explanation_signal'],
    'top_decision_hint': view['model_difference_summary']['top_decision_hint'],
    'top_segment': view['model_difference_summary']['top_segment'],
    'has_difference_panel': 'Model Difference Explanations' in html,
    'has_difference_signal_filter': 'explanation-signal-filter' in html,
    'has_difference_decision_filter': 'difference-decision-filter' in html,
    'html_path': 'dashboard.html',
    'dashboard_export_path': 'dashboard_export.json',
    'dashboard_view_model_path': 'dashboard_view_model.json',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.dashboard_model_difference_explanations.v1',
 'status': 'completed',
 'source_card': 'research/quantbench_model_matrix_smoke_v1',
 'difference_source_card': 'research/analysis_model_difference_explanations_v1',
 'model_card_count': 12,
 'difference_explanation_count': 19,
 'top_explanation_signal': 'segment_winner',
 'top_decision_hint': 'promote_segment_candidate',
 'top_segment': 'late_test::downtrend',
 'has_difference_panel': True,
 'has_difference_signal_filter': True,
 'has_difference_decision_filter': True,
 'html_path': 'dashboard.html',
 'dashboard_export_path': 'dashboard_export.json',
 'dashboard_view_model_path': 'dashboard_view_model.json'}

In [3]:
preview = pd.DataFrame(view['model_difference_explanations']).loc[:, [
    'priority_rank',
    'segment_type',
    'segment',
    'winner_model',
    'loser_model',
    'relative_return_pct_spread',
    'explanation_signal',
    'decision_hint',
    'review_note',
]]
preview.head(10)

,priority_rank,segment_type,segment,winner_model,loser_model,relative_return_pct_spread,explanation_signal,decision_hint,review_note
0,1,window_regime,late_test::downtrend,alstm_mse,logistic_clf,47.899136,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 late_test::downtrend 强于基...
1,2,regime,downtrend,alstm_mse,mlp_mse,52.699142,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 downtrend 强于基准 logistic_...
2,3,window_regime,full_test::downtrend,alstm_mse,mlp_mse,52.699142,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 full_test::downtrend 强于基...
3,4,all,all,logistic_clf,mlp_mse,42.555385,baseline_defends,block_or_rework_loser,中文 review：基准 logistic_clf 在 all 领先，mlp_mse 是当前...
4,5,window,full_test,logistic_clf,mlp_mse,42.555385,baseline_defends,block_or_rework_loser,中文 review：基准 logistic_clf 在 full_test 领先，mlp_m...
5,6,window,late_test,extra_trees_rank,mlp_mse,34.496111,segment_winner,promote_segment_candidate,中文 review：extra_trees_rank 在 late_test 强于基准 lo...
6,7,window_regime,late_test::bull,alstm_mse,mlp_mse,36.832691,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 late_test::bull 强于基准 log...
7,8,regime,stress,extra_trees_rank,alstm_mse,33.871405,segment_winner,promote_segment_candidate,中文 review：extra_trees_rank 在 stress 强于基准 logis...
8,9,window_regime,full_test::stress,extra_trees_rank,alstm_mse,33.871405,segment_winner,promote_segment_candidate,中文 review：extra_trees_rank 在 full_test::stress...
9,10,regime,bull,catboost_rank,alstm_mse,24.656980,segment_winner,promote_segment_candidate,中文 review：catboost_rank 在 bull 强于基准 logistic_c...


In [4]:
conclusion_lines = [
    '## 中文结论',
    '',
    '- 做了什么：本 notebook 将 `model_difference_explanations.csv` 接入 `src/dashboard`，生成 dashboard export、view model 和静态 HTML。',
    f"- 核心指标：dashboard 有 `{summary['model_card_count']}` 个模型卡，差异解释 `{summary['difference_explanation_count']}` 条；top signal=`{summary['top_explanation_signal']}`，top decision=`{summary['top_decision_hint']}`。",
    f"- regime 差异：最高优先级分段是 `{summary['top_segment']}`，HTML 中已经有 `Model Difference Explanations` 面板、explanation signal filter 和 difference decision filter。",
    '- 是否晋升：不晋升。本实验只验证 dashboard 展示链路，不证明任何 QuantBench smoke 模型可进入 production。',
    '- 风险：输入来自 smoke 回测和研究解释表，样本短；dashboard 只展示 review 线索，不能替代 objective、backtest、production health 和 paper execution gate。',
    '- 下一步：把该面板纳入 schedule refresh 产物，让定时 dashboard export 自动携带模型差异解释。',
]
conclusion = '\n'.join(conclusion_lines)
(CARD / 'analysis_conclusion.md').write_text(conclusion + '\n', encoding='utf-8')
CARD / 'analysis_conclusion.md'

PosixPath('/Users/rziiiii/dev/workspace/research/dashboard_model_difference_explanations_v1/analysis_conclusion.md')

## 中文结论

- 做了什么：本 notebook 将 `model_difference_explanations.csv` 接入 `src/dashboard`，生成 dashboard export、view model 和静态 HTML。
- 核心指标：dashboard 有 `12` 个模型卡，差异解释 `19` 条；top signal=`segment_winner`，top decision=`promote_segment_candidate`。
- regime 差异：最高优先级分段是 `late_test::downtrend`，HTML 中已经有 `Model Difference Explanations` 面板、explanation signal filter 和 difference decision filter。
- 是否晋升：不晋升。本实验只验证 dashboard 展示链路，不证明任何 QuantBench smoke 模型可进入 production。
- 风险：输入来自 smoke 回测和研究解释表，样本短；dashboard 只展示 review 线索，不能替代 objective、backtest、production health 和 paper execution gate。
- 下一步：把该面板纳入 schedule refresh 产物，让定时 dashboard export 自动携带模型差异解释。
